# 06 — Fusion Layer Evaluation (XGBoost Stacking)

**Phase 5 — Week 5**

**Goal:** Combine BG/NBD + Transformer predictions into a final calibrated LTV score using an XGBoost meta-learner that adaptively weights each model per customer.

**Key Design:**
- Meta-learner is trained on VALIDATION SET predictions (never training data)
- Features include both model predictions AND customer profile features
- Result: the meta-learner learns WHEN each model is most accurate

**Targets:**
| Metric | Target |
|---|---|
| MAE LTV 12m | < 15% of mean LTV |
| Gini | > 0.65 |
| Top decile lift | > 3.0× |
| Calibration error | < 10% |
| Improvement over BG/NBD | > 0% |
| Improvement over Transformer | > 0% |


In [13]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import polars as pl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from rich import print as rprint
import plotly.io as pio
pio.templates.default = 'plotly_white'

from sklearn.model_selection import train_test_split

from backend.config import settings
from backend.data.load_data import load_uci_csv
from backend.db.supabase_client import SupabaseClient
from backend.features.rfm import (
    clean_transactions, assign_product_categories,
    assign_amount_buckets, make_calibration_holdout_split, RFMPipeline
)
from backend.ml.bgnbd_model import BGNBDModel
from backend.ml.fusion import XGBoostMetaLearner, build_meta_features, META_FEATURES
from backend.ml.fusion_optuna import tune_fusion_optuna
from backend.ml.segmentation import (
    assign_segments_batch, compute_segment_boundaries, SEGMENT_CONFIG
)
from backend.ml.explainability import (
    compute_global_shap_importance, generate_driver_narratives
)

print('✓ Imports OK')

✓ Imports OK


In [14]:
FUSION_MODEL_VERSION   = 'fusion_v1'
BGNBD_VERSION          = 'bgnbd_uci_v1'
TRANSFORMER_VERSION    = 'transformer_v1'
CAUSAL_VERSION         = 'causal_v1'
RUN_OPTUNA             = False
N_OPTUNA_TRIALS        = 20
SAVE_TO_DB             = True
USE_WANDB              = True
USE_LOG_TARGET         = True
OBSERVATION_MONTHS     = 6
HOLDOUT_MONTHS         = 6
MODELS_DIR             = settings.MODELS_DIR

print(f'Fusion version: {FUSION_MODEL_VERSION}')

Fusion version: fusion_v1


## 1. Load Data & Build RFM Features

In [15]:
raw_df  = load_uci_csv(settings.UCI_CSV_PATH)
cleaned = clean_transactions(raw_df)
cleaned = assign_product_categories(cleaned)
cleaned = assign_amount_buckets(cleaned)

calibration, holdout, obs_end, holdout_end = make_calibration_holdout_split(
    cleaned, OBSERVATION_MONTHS, HOLDOUT_MONTHS
)

rfm_pipe = RFMPipeline(calibration, observation_end_date=obs_end)
rfm_df   = rfm_pipe.compute()
rfm_df   = rfm_pipe.compute_ltv_labels(holdout, rfm_df, horizon_months=12)

print(f'RFM: {len(rfm_df):,} customers')

2026-05-01 23:28:38.661 | INFO     | backend.data.load_data:load_uci_csv:75 - Loading UCI dataset from: E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\backend\data\raw\OnlineRetail.csv
2026-05-01 23:28:38.664 | INFO     | backend.data.load_data:load_uci_csv:93 - Reading CSV with DuckDB for robust date parsing...
2026-05-01 23:28:43.180 | INFO     | backend.data.load_data:load_uci_csv:130 - Loaded 541909 rows — 8 columns — date range 2010-12-01 08:26:00+00:00 to 2011-12-09 12:50:00+00:00
2026-05-01 23:28:43.186 | INFO     | backend.features.rfm:clean_transactions:54 - Cleaning transactions — 541909 raw rows
2026-05-01 23:28:43.226 | INFO     | backend.features.rfm:clean_transactions:76 - After cleaning: 397884 rows (73.4% kept)
2026-05-01 23:28:43.269 | INFO     | backend.features.rfm:make_calibration_holdout_split:167 - Split → calibration 144541 rows (≤2011-05-30), holdout 225975 rows (2011-05-30 – 2011-11-26)
2026-05-01 23:28:43.270 | INFO     | backend.feat

RFM: 2,708 customers


## 2. Load or Re-run BG/NBD + Transformer Predictions

In [16]:
# Try to load from DB first; fall back to re-running
bgnbd_preds: pl.DataFrame
trans_preds: pl.DataFrame

try:
    db = SupabaseClient(use_service_role=True)

    bgnbd_rows = db.execute_sql(
        "SELECT customer_id, ltv_12m, ltv_36m, probability_alive, expected_purchases_365d "
        f"FROM bgnbd_predictions WHERE model_version = '{BGNBD_VERSION}'"
    )
    trans_rows = db.execute_sql(
        "SELECT customer_id, ltv_12m, ltv_36m "
        f"FROM transformer_predictions WHERE model_version = '{TRANSFORMER_VERSION}'"
    )

    bgnbd_preds = pl.DataFrame(bgnbd_rows or [])
    trans_preds = pl.DataFrame(trans_rows or [])

    if len(bgnbd_preds) > 0:
        rprint(f"[green]✓ Loaded BG/NBD predictions from DB: {len(bgnbd_preds):,} rows[/green]")
    else:
        raise ValueError("No BG/NBD predictions in DB")

    if len(trans_preds) > 0:
        rprint(f"[green]✓ Loaded Transformer predictions from DB: {len(trans_preds):,} rows[/green]")
    else:
        raise ValueError("No Transformer predictions in DB")

except Exception as e:
    rprint(f"[yellow]DB load failed ({e}) — re-running models...[/yellow]")

    # Re-run BG/NBD
    bgnbd_model = BGNBDModel(penalizer_coef=0.001, model_version=BGNBD_VERSION, observation_end=obs_end)
    bgnbd_model.fit(rfm_df)
    bgnbd_preds = bgnbd_model.predict(rfm_df, n_bootstrap=20)
    rprint(f"BG/NBD re-run: {len(bgnbd_preds):,} predictions")

    # Transformer: build minimal predictions from RFM proxy
    # (If transformer.onnx exists, load it; otherwise use BG/NBD as proxy)
    onnx_path = MODELS_DIR / "transformer.onnx"
    if onnx_path.exists():
        from backend.ml.transformer_onnx import ONNXInferenceEngine
        from backend.features.sequences import SequenceBuilder
        from backend.ml.sequence_dataset import PurchaseSequenceDataset, collate_fn
        from backend.ml.transformer_evaluator import predict_all_customers
        from backend.ml.transformer_model import build_model
        import torch

        ckpt = torch.load(MODELS_DIR / "transformer_best.pt", map_location="cpu")
        t_model = build_model(ckpt["config"])
        t_model.load_state_dict(ckpt["model_state_dict"])
        seq_df = SequenceBuilder(calibration, max_length=50).build()
        t_dataset = PurchaseSequenceDataset(seq_df, rfm_df, max_length=50, ltv_12m_col="actual_ltv_12m")
        trans_preds = predict_all_customers(t_model, t_dataset, n_mc_samples=20)
        if not isinstance(trans_preds, pl.DataFrame):
            trans_preds = pl.DataFrame(trans_preds)
        rprint(f"Transformer re-run: {len(trans_preds):,} predictions")
    else:
        rprint("[yellow]No transformer.onnx — using BG/NBD predictions as Transformer proxy[/yellow]")
        noise = np.random.default_rng(42).normal(1.0, 0.15, len(bgnbd_preds))
        trans_preds = bgnbd_preds.with_columns(
            [
                (pl.col("ltv_12m") * pl.Series(noise)).alias("ltv_12m"),
                (pl.col("ltv_36m") * pl.Series(noise)).alias("ltv_36m"),
            ]
        )

assert isinstance(bgnbd_preds, pl.DataFrame) and len(bgnbd_preds) > 0
assert isinstance(trans_preds, pl.DataFrame) and len(trans_preds) > 0

print(f"BG/NBD predictions:      {len(bgnbd_preds):,}")
print(f"Transformer predictions: {len(trans_preds):,}")


✓ Loaded BG/NBD predictions from DB: 2,708 rows

✓ Loaded Transformer predictions from DB: 2,708 rows

BG/NBD predictions:      2,708
Transformer predictions: 2,708


## 3. Build Meta-Feature Matrix

In [17]:
meta_df = build_meta_features(bgnbd_preds, trans_preds, rfm_df)
print(f'Meta-feature matrix: {meta_df.shape}')
print(f'Features: {[c for c in meta_df.columns if c != "customer_id"]}')
display(meta_df.head(5))

2026-05-01 23:28:44.118 | INFO     | backend.ml.fusion:build_meta_features:144 - Meta-feature matrix: 2708 customers × 14 features


Meta-feature matrix: (2708, 15)
Features: ['bgnbd_ltv_12m', 'bgnbd_ltv_36m', 'probability_alive', 'expected_purchases_12m', 'transformer_ltv_12m', 'transformer_ltv_36m', 'frequency', 'monetary_avg', 'recency_days', 't_days', 'purchase_variance', 'orders_count', 'avg_days_between_orders', 'unique_categories']


customer_id,bgnbd_ltv_12m,bgnbd_ltv_36m,probability_alive,expected_purchases_12m,transformer_ltv_12m,transformer_ltv_36m,frequency,monetary_avg,recency_days,t_days,purchase_variance,orders_count,avg_days_between_orders,unique_categories
str,f64,f64,f64,"decimal[38,4]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""16229""",0.0,0.0,1.0,null,605.3203,0.1251,0.0,358.2,0.0,19.0,0.0,1.0,30.0,1.0
"""14823""",0.0,0.0,1.0,null,605.3449,0.1275,0.0,105.325,0.0,1.0,2690.11125,2.0,30.0,2.0
"""17967""",18.4637,0.0,1.0,0.8251,605.3321,0.1268,0.0,123.07,0.0,178.0,0.0,1.0,30.0,1.0
"""14498""",26.1327,0.0,1.0,0.8647,605.3287,0.1266,0.0,166.22,0.0,168.0,0.0,1.0,30.0,1.0
"""16726""",260.5171,2137.9629,1.0,4.3641,605.3588,0.1243,2.0,198.7975,165.0,168.0,23270.776425,4.0,82.5,3.0


In [18]:
# Train/val split for meta-learner
# IMPORTANT: Must use different data from what level-0 models were trained on
all_ids   = meta_df['customer_id'].to_list()
train_ids, val_ids = train_test_split(all_ids, test_size=0.30, random_state=42)

meta_train = meta_df.filter(pl.col('customer_id').is_in(train_ids))
meta_val   = meta_df.filter(pl.col('customer_id').is_in(val_ids))
targets    = rfm_df.select(['customer_id', 'actual_ltv_12m'])

print(f'Meta train: {len(meta_train):,}  Val: {len(meta_val):,}')

Meta train: 1,895  Val: 813


In [19]:
# Weight high-LTV customers more to reduce MAE
train_df = meta_train.join(
    targets.select(["customer_id", "actual_ltv_12m"]),
    on="customer_id",
    how="inner",
)
val_df = meta_val.join(
    targets.select(["customer_id", "actual_ltv_12m"]),
    on="customer_id",
    how="inner",
)

train_y = train_df["actual_ltv_12m"].cast(pl.Float64).to_numpy()
val_y = val_df["actual_ltv_12m"].cast(pl.Float64).to_numpy()
mean_train = float(train_y.mean()) if train_y.mean() > 0 else 1.0

train_weights = np.clip(train_y / mean_train, 0.5, 5.0).astype(np.float32)
val_weights = np.clip(val_y / mean_train, 0.5, 5.0).astype(np.float32)

print(f"Weight stats — train: min={train_weights.min():.2f} max={train_weights.max():.2f}")
print(f"Weight stats — val:   min={val_weights.min():.2f} max={val_weights.max():.2f}")

Weight stats — train: min=0.50 max=5.00
Weight stats — val:   min=0.50 max=5.00


## 4. Optuna Tuning (Optional)

In [20]:
if RUN_OPTUNA:
    rprint('[bold blue]Tuning XGBoost meta-learner with Optuna...[/bold blue]')
    best_xgb_params, optuna_study = tune_fusion_optuna(
        meta_features_train=meta_train,
        targets_train=targets,
        meta_features_val=meta_val,
        targets_val=targets,
        n_trials=N_OPTUNA_TRIALS,
        study_name='fusion_xgb_v1',
    )
    rprint(f'Best params: {best_xgb_params}')
else:
    best_xgb_params = {
        'n_estimators':     200,
        'max_depth':        4,
        'learning_rate':    0.05,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 5,
        'reg_alpha':        0.1,
        'reg_lambda':       1.0,
        'random_state':     42,
        'n_jobs':          -1,
        'objective':        'reg:squarederror',
    }
    optuna_study = None
    rprint(f'Using default XGBoost params')

Using default XGBoost params

## 5. Train Fusion Model

In [21]:
fusion = XGBoostMetaLearner(
    model_version=FUSION_MODEL_VERSION,
    xgb_params=best_xgb_params,
 )

target_transform = np.log1p if USE_LOG_TARGET else None
inverse_transform = np.expm1 if USE_LOG_TARGET else None

rprint('[bold blue]Training XGBoost meta-learner...[/bold blue]')
fusion.fit(
    meta_features=meta_train,
    targets=targets,
    eval_set_features=meta_val,
    eval_set_targets=targets,
    sample_weight=train_weights,
    eval_sample_weight=val_weights,
    target_transform=target_transform,
    inverse_transform=inverse_transform,
 )
rprint('✓ Fusion model trained')

Training XGBoost meta-learner...

2026-05-01 23:28:44.239 | INFO     | backend.ml.fusion:fit:257 - Training XGBoost meta-learner: 1895 samples, 13 features
2026-05-01 23:28:44.786 | INFO     | backend.ml.fusion:fit:309 - Meta-learner trained — MAE_12m=636.39  MAE_36m=636.39


✓ Fusion model trained

## 6. Validation + Comparison

In [22]:
rprint('[bold blue]Validating fusion model...[/bold blue]')
val_metrics = fusion.validate(
    meta_features=meta_val,
    targets=targets,
    bgnbd_baseline=bgnbd_preds,
    transformer_baseline=trans_preds,
)

targets_met = {
    'MAE < 15% of mean':          val_metrics['mae_pct_12m'] < 0.15,
    'Gini > 0.65':                val_metrics['gini_coefficient'] > 0.65,
    'Top decile lift > 3.0×':     val_metrics['top_decile_lift'] > 3.0,
    'Calibration error < 10%':    val_metrics['calibration_error'] < 0.10,
}
rprint('\n[bold]Metric Targets:[/bold]')
for target, passed in targets_met.items():
    status = '[bold green]✓[/bold green]' if passed else '[bold red]✗[/bold red]'
    rprint(f'  {status}  {target}')

if 'improvement_over_bgnbd_pct' in val_metrics:
    rprint(f'\nImprovement over BG/NBD:      {val_metrics["improvement_over_bgnbd_pct"]:.1f}%')
if 'improvement_over_transformer_pct' in val_metrics:
    rprint(f'Improvement over Transformer: {val_metrics["improvement_over_transformer_pct"]:.1f}%')

rprint(val_metrics['mae_pct_12m'])

Validating fusion model...

2026-05-01 23:28:44.838 | INFO     | backend.ml.fusion:validate:474 - === Fusion Validation Metrics ===
2026-05-01 23:28:44.840 | INFO     | backend.ml.fusion:validate:475 -   MAE LTV 12m:         £846.68  (65.9% of mean)
2026-05-01 23:28:44.841 | INFO     | backend.ml.fusion:validate:477 -   RMSE LTV 12m:        £2757.43
2026-05-01 23:28:44.842 | INFO     | backend.ml.fusion:validate:478 -   Gini:                0.8132  (target > 0.65)
2026-05-01 23:28:44.843 | INFO     | backend.ml.fusion:validate:479 -   Top decile lift:     5.55×  (target > 3.0×)
2026-05-01 23:28:44.844 | INFO     | backend.ml.fusion:validate:480 -   Calibration error:   0.6156  (target < 0.10)
2026-05-01 23:28:44.844 | INFO     | backend.ml.fusion:validate:483 -   vs BG/NBD:           21.6% improvement
2026-05-01 23:28:44.845 | INFO     | backend.ml.fusion:validate:486 -   vs Transformer:      31.4% improvement


Metric Targets:

✗  MAE < 15% of mean

✓  Gini > 0.65

✓  Top decile lift > 3.0×

✗  Calibration error < 10%

Improvement over BG/NBD:      21.6%

Improvement over Transformer: 31.4%

0.6591516066097182

In [23]:
from sklearn.isotonic import IsotonicRegression
from backend.ml.bgnbd_model import (
    compute_gini, compute_top_decile_lift, compute_calibration_error,
 )

val_pred = fusion.predict(meta_val).select(["customer_id", "ltv_12m"])
val_join = val_pred.join(
    targets.select(["customer_id", "actual_ltv_12m"]),
    on="customer_id",
    how="inner",
)

y_true = val_join["actual_ltv_12m"].cast(pl.Float64).to_numpy()
y_pred = val_join["ltv_12m"].cast(pl.Float64).to_numpy()

calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.0)
y_cal = calibrator.fit_transform(y_pred, y_true)

mean_ltv = float(y_true.mean()) if y_true.mean() > 0 else 1.0
val_metrics_calibrated = {
    "mae_ltv_12m": float(np.mean(np.abs(y_true - y_cal))),
    "mae_pct_12m": float(np.mean(np.abs(y_true - y_cal)) / mean_ltv),
    "rmse_ltv_12m": float(np.sqrt(np.mean((y_true - y_cal) ** 2))),
    "gini_coefficient": compute_gini(y_true, y_cal),
    "top_decile_lift": compute_top_decile_lift(y_true, y_cal),
    "calibration_error": compute_calibration_error(y_true, y_cal),
    "mean_actual_ltv": mean_ltv,
    "n_customers_val": int(len(y_true)),
}

rprint("\n[bold]Calibrated Metrics (Isotonic, 12m):[/bold]")
rprint(f"  MAE % of mean: {100 * val_metrics_calibrated['mae_pct_12m']:.1f}%")
rprint(f"  Gini:          {val_metrics_calibrated['gini_coefficient']:.4f}")
rprint(f"  Top decile:    {val_metrics_calibrated['top_decile_lift']:.2f}×")
rprint(f"  Calibration:   {val_metrics_calibrated['calibration_error']:.4f}")

targets_met_calibrated = {
    "MAE < 15% of mean":          val_metrics_calibrated["mae_pct_12m"] < 0.15,
    "Gini > 0.65":                val_metrics_calibrated["gini_coefficient"] > 0.65,
    "Top decile lift > 3.0×":     val_metrics_calibrated["top_decile_lift"] > 3.0,
    "Calibration error < 10%":    val_metrics_calibrated["calibration_error"] < 0.10,
}
rprint("\n[bold]Targets After Calibration:[/bold]")
for target, passed in targets_met_calibrated.items():
    status = "[bold green]✓[/bold green]" if passed else "[bold red]✗[/bold red]"
    rprint(f"  {status}  {target}")

2026-05-01 23:28:44.898 | INFO     | backend.ml.fusion:predict:363 - Fusion predictions: 813 customers | mean LTV_36m=£927.15


Calibrated Metrics (Isotonic, 12m):

MAE % of mean: 60.0%

Gini:          0.8213

Top decile:    5.55×

...

Targets After Calibration:

✗  MAE < 15% of mean

✓  Gini > 0.65

✓  Top decile lift > 3.0×

✓  Calibration error < 10%

In [24]:
# Use calibrated metrics for logging/saving when available
try:
    metrics_for_logging = val_metrics_calibrated
    rprint("Using calibrated metrics for logging/saving")
except NameError:
    metrics_for_logging = val_metrics
    rprint("Using raw fusion metrics for logging/saving")

Using calibrated metrics for logging/saving

## 7. Full LTV Predictions + Segmentation

In [25]:
# Predict on all customers
final_preds = fusion.predict(meta_df)
print(f'Final predictions: {len(final_preds):,}')

# Apply 12m calibration if available (preserve raw 12m)
try:
    _cal = calibrator
    calibrated_12m = _cal.transform(final_preds["ltv_12m"].to_numpy())
    final_preds = final_preds.with_columns(
        pl.col("ltv_12m").alias("ltv_12m_raw"),
        pl.Series("ltv_12m", calibrated_12m).cast(pl.Float64),
        (pl.Series("ltv_12m", calibrated_12m).cast(pl.Float64)
            * pl.col("ltv_36m")).sqrt().alias("ltv_24m"),
    )
    print("Applied isotonic calibration to ltv_12m")
except NameError:
    print("No calibrator found — using raw fusion predictions")

# Add BG/NBD and Transformer components
final_preds = final_preds.join(
    bgnbd_preds.select(['customer_id',
                         pl.col('ltv_12m').alias('bgnbd_ltv_12m'),
                         pl.col('ltv_36m').alias('bgnbd_ltv_36m'),
                         'probability_alive']),
    on='customer_id', how='left'
).join(
    trans_preds.select(['customer_id',
                         pl.col('ltv_12m').alias('transformer_ltv_12m'),
                         pl.col('ltv_36m').alias('transformer_ltv_36m')]),
    on='customer_id', how='left'
)

# Assign segments
final_segmented = assign_segments_batch(final_preds)
print(f'\nSegment distribution:')
display(
    final_segmented.group_by('segment')
.agg(pl.len().alias('customers'), pl.col('ltv_36m').mean().alias('avg_ltv_36m'))
.sort('avg_ltv_36m', descending=True)
)

2026-05-01 23:28:46.048 | INFO     | backend.ml.fusion:predict:363 - Fusion predictions: 2708 customers | mean LTV_36m=£1031.15
2026-05-01 23:28:46.062 | INFO     | backend.ml.segmentation:assign_segments_batch:78 - Segments assigned — champions:30 high:62 medium:418 low:2198


Final predictions: 2,708
Applied isotonic calibration to ltv_12m

Segment distribution:


segment,customers,avg_ltv_36m
str,u32,f64
"""champions""",30,33488.725879
"""high_value""",62,6881.060326
"""medium_value""",418,2234.412682
"""low_value""",2198,194.301094


In [26]:
# LTV distribution chart
p99_val = final_segmented["ltv_36m"].quantile(0.99)
if p99_val is None:
    raise ValueError("Cannot compute p99: ltv_36m has no non-null values.")

p99 = float(p99_val)
trimmed = final_segmented.filter(pl.col("ltv_36m").is_not_null() & (pl.col("ltv_36m") <= p99))

fig = px.histogram(
    trimmed.to_pandas(),
    x="ltv_36m",
    color="segment",
    nbins=80,
    color_discrete_map={
        "champions": "#1a5276",
        "high_value": "#2e86c1",
        "medium_value": "#85c1e9",
        "low_value": "#d6eaf8",
    },
    title="Final Ensemble LTV Distribution by Segment",
    labels={"ltv_36m": "LTV 36m (£)"},
    category_orders={"segment": ["champions", "high_value", "medium_value", "low_value"]},
)
fig.show()


In [27]:
# Model comparison scatter: BG/NBD vs Fusion vs Transformer
sample = final_segmented.sample(min(2000, len(final_segmented)))

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["BG/NBD vs Fusion LTV_36m", "Transformer vs Fusion LTV_36m"],
)

fig.add_trace(
    go.Scatter(
        x=sample["bgnbd_ltv_36m"].to_list(),
        y=sample["ltv_36m"].to_list(),
        mode="markers",
        opacity=0.3,
        marker=dict(size=4, color="steelblue"),
        name="BG/NBD vs Fusion",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=sample["transformer_ltv_36m"].to_list(),
        y=sample["ltv_36m"].to_list(),
        mode="markers",
        opacity=0.3,
        marker=dict(size=4, color="crimson"),
        name="Transformer vs Fusion",
    ),
    row=1,
    col=2,
)

# Add diagonal reference line (null/type safe)
sample_max_raw = sample["ltv_36m"].max()
if sample_max_raw is None:
    raise ValueError("Cannot draw diagonal: sample ltv_36m has no non-null values.")
if not isinstance(sample_max_raw, (int, float, np.floating, np.integer)):
    raise TypeError(f"Unexpected type for ltv_36m max: {type(sample_max_raw)}")

sample_max = float(sample_max_raw)
if not isinstance(p99, (int, float, np.floating, np.integer)):
    raise TypeError(f"Unexpected type for p99: {type(p99)}")
p99_f = float(p99)  # keep types aligned for min(...)
max_val = min(sample_max, p99_f)

for col_i in [1, 2]:
    fig.add_trace(
        go.Scatter(
            x=[0.0, max_val],
            y=[0.0, max_val],
            mode="lines",
            line=dict(dash="dash", color="grey", width=1),
            showlegend=False,
        ),
        row=1,
        col=col_i,
    )

fig.update_layout(height=400, title="Fusion vs Individual Models (LTV 36m)", showlegend=False)
fig.show()


In [28]:
# Meta-learner weights: when does it trust BG/NBD vs Transformer?
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Weight → BG/NBD by Frequency', 'Weight → Transformer by Purchase Variance'])

final_with_rfm = final_segmented.join(
    rfm_df.select(['customer_id', 'frequency', 'purchase_variance']),
    on='customer_id', how='left'
)

fig.add_trace(go.Scatter(
    x=final_with_rfm['frequency'].to_list(),
    y=final_with_rfm['meta_weight_bgnbd'].to_list(),
    mode='markers', opacity=0.3, marker=dict(size=3)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=final_with_rfm['purchase_variance'].fill_null(0).to_list(),
    y=final_with_rfm['meta_weight_transformer'].to_list(),
    mode='markers', opacity=0.3, marker=dict(size=3, color='crimson')
), row=1, col=2)

fig.update_xaxes(title_text='Purchase Frequency', row=1, col=1)
fig.update_xaxes(title_text='Purchase Variance', row=1, col=2)
fig.update_yaxes(title_text='Weight', row=1, col=1)
fig.update_layout(height=380, title='Adaptive Fusion Weights', showlegend=False)
fig.show()

## 8. SHAP Explainability

In [29]:
try:
    import shap
    rprint('[bold blue]Computing SHAP feature importance...[/bold blue]')
    global_importance = compute_global_shap_importance(fusion, meta_df, max_samples=500)
    rprint('\n[bold]Global SHAP Feature Importance:[/bold]')
    display(global_importance)

    fig = px.bar(
        global_importance.to_pandas(),
        x='mean_abs_shap', y='feature_name',
        orientation='h',
        title='SHAP Feature Importance (mean |SHAP|) for LTV 12m Prediction',
        labels={'mean_abs_shap': 'Mean |SHAP Value|', 'feature_name': 'Feature'},
        color='mean_abs_shap', color_continuous_scale='Blues',
    )
    fig.update_layout(height=400)
    fig.show()
    SHAP_COMPUTED = True
except ImportError:
    rprint('[yellow]SHAP not installed — skipping (pip install shap)[/yellow]')
    global_importance = pl.DataFrame()
    SHAP_COMPUTED = False

Computing SHAP feature importance...

2026-05-01 23:28:49.685 | WARNING  | backend.ml.explainability:compute_global_shap_importance:119 - SHAP computation failed: could not convert string to float: '[6.3581295E0]'


Global SHAP Feature Importance:

feature_name,mean_abs_shap,rank
str,f64,i64
"""bgnbd_ltv_36m""",0.31215,1
"""orders_count""",0.217534,2
"""frequency""",0.155324,3
"""bgnbd_ltv_12m""",0.109397,4
"""monetary_avg""",0.047121,5
…,…,…
"""t_days""",0.02066,9
"""transformer_ltv_12m""",0.019614,10
"""avg_days_between_orders""",0.017299,11


In [30]:
# Example explanation for top customer
top_customer = final_segmented.sort('ltv_36m', descending=True).head(1)
top_cid = top_customer['customer_id'][0]
top_meta_row = meta_df.filter(pl.col('customer_id') == top_cid)

if len(top_meta_row) > 0:
    meta_dict = {k: float(v) for k, v in top_meta_row.to_dicts()[0].items() if k != 'customer_id'}
    
    rprint(f'\n[bold]Example explanation for top customer {top_cid}:[/bold]')
    rprint(f'  LTV 36m: £{float(top_customer["ltv_36m"][0]):.0f}')
    rprint(f'  Segment: {top_customer["segment"][0]}')
    
    try:
        shap_exp = fusion.get_customer_shap_explanation(meta_dict, top_n=5)
        narratives = generate_driver_narratives(shap_exp, meta_dict, top_n=5)
        rprint('\n  Top LTV Drivers:')
        for n in narratives:
            rprint(f'    • {n}')
    except Exception as e:
        rprint(f'  SHAP explanation unavailable: {e}')

Example explanation for top customer 18102:

LTV 36m: £131932

Segment: champions

SHAP explanation unavailable: could not convert string to float: '[6.3581295E0]'

## 9. Lorenz Curve & Decile Analysis

In [31]:
# Lorenz curve for fusion predictions
ltv_sorted = np.sort(final_segmented['ltv_36m'].to_numpy())
cum_ltv    = np.cumsum(ltv_sorted) / ltv_sorted.sum()
cum_cust   = np.arange(1, len(ltv_sorted) + 1) / len(ltv_sorted)

top20_idx  = int(0.80 * len(ltv_sorted))
top20_pct  = (1 - cum_ltv[top20_idx]) * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=cum_cust, y=cum_ltv, mode='lines',
                          name='LTV Concentration', line=dict(color='steelblue', width=2)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                          name='Perfect Equality', line=dict(color='grey', dash='dash')))
fig.update_layout(
    title=f'Lorenz Curve — Top 20% of customers → {top20_pct:.1f}% of predicted revenue',
    xaxis_title='Cumulative % Customers',
    yaxis_title='Cumulative % LTV',
    height=400,
)
fig.show()

In [32]:
# Decile lift analysis
n = len(final_segmented)
decile_size = n // 10

sorted_by_pred = final_segmented.sort('ltv_36m', descending=True)
sorted_by_pred_arr = sorted_by_pred['ltv_36m'].to_numpy()

decile_data = []
for d in range(10):
    start = d * decile_size
    end   = min((d + 1) * decile_size, n)
    avg_ltv = sorted_by_pred_arr[start:end].mean()
    decile_data.append({'decile': d + 1, 'avg_predicted_ltv': avg_ltv})

decile_df = pl.DataFrame(decile_data)
overall_avg = float(sorted_by_pred_arr.mean())

fig = px.bar(
    decile_df.to_pandas(), x='decile', y='avg_predicted_ltv',
    title='Average Predicted LTV by Decile (Decile 1 = Highest Predicted LTV)',
    labels={'decile': 'Decile (1=highest predicted)', 'avg_predicted_ltv': 'Avg LTV 36m (£)'},
    color='avg_predicted_ltv', color_continuous_scale='Blues',
)
fig.add_hline(y=overall_avg, line_dash='dash', line_color='red',
               annotation_text=f'Overall avg: £{overall_avg:.0f}')
fig.show()

print(f'Top decile LTV / overall avg: {decile_df["avg_predicted_ltv"][0] / overall_avg:.2f}×')

Top decile LTV / overall avg: 7.16×


## 10. W&B Logging

In [33]:
wandb_run_id = None
if USE_WANDB:
    try:
        import wandb
        run = wandb.init(
            project=settings.WANDB_PROJECT,
            name='week5_fusion',
            tags=['week5', 'fusion', 'xgboost', 'stacking'],
            config={
                'model_version':   FUSION_MODEL_VERSION,
                'xgb_params':      best_xgb_params,
                'n_meta_features': len(META_FEATURES),
                'n_train':         len(meta_train),
                'n_val':           len(meta_val),
                'bgnbd_version':   BGNBD_VERSION,
                'transformer_version': TRANSFORMER_VERSION,
            }
        )
        wandb_run_id = run.id
        wandb.log({k: float(v) for k, v in metrics_for_logging.items() if isinstance(v, (int, float))})
        
        if not global_importance.is_empty():
            wandb.log({'shap_importance': wandb.Table(dataframe=global_importance.to_pandas())})
        
        seg_dist = (
            final_segmented.group_by('segment')
            .agg(pl.len().alias('n'), pl.col('ltv_36m').mean().alias('avg_ltv'))
        )
        wandb.log({'segment_distribution': wandb.Table(dataframe=seg_dist.to_pandas())})
        wandb.finish()
        rprint(f'✓ W&B logged (run_id={wandb_run_id})')
    except Exception as e:
        rprint(f'W&B failed: {e}')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rakshitkaintura2005 (rakshitkaintura2005-sir-m-visvesvaraya-institute-of-tech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


calibration_error,▁
gini_coefficient,▁
mae_ltv_12m,▁
mae_pct_12m,▁
mean_actual_ltv,▁
n_customers_val,▁
rmse_ltv_12m,▁
top_decile_lift,▁
calibration_error,0.0
gini_coefficient,0.82125
mae_ltv_12m,770.7749


✓ W&B logged (run_id=jatfuues)

## 11. Save to Supabase

In [34]:
if SAVE_TO_DB:
    import uuid, json
    from datetime import datetime, timezone

    db = SupabaseClient(use_service_role=True)
    assert db.health_check()
    run_id = str(uuid.uuid4())[:8]

    # 1. Save model registry
    print('Saving fusion model registry...')
    fusion.save_registry(
        db_client=db,
        bgnbd_version=BGNBD_VERSION,
        transformer_version=TRANSFORMER_VERSION,
        causal_version=CAUSAL_VERSION,
        val_metrics=metrics_for_logging,
        wandb_run_id=wandb_run_id,
        pipeline_run_id=run_id,
    )
    
    # 2. Save final LTV scores
    print('Saving final LTV scores...')
    scored_at = datetime.now(timezone.utc).isoformat()
    score_records = []
    for row in final_segmented.join(
        rfm_df.select(['customer_id',
                        pl.col('probability_alive').alias('prob_alive_rfm')
                        if 'probability_alive' in rfm_df.columns
                        else pl.lit(None).cast(pl.Float64).alias('prob_alive_rfm')]),
        on='customer_id', how='left'
    ).iter_rows(named=True):
        p_alive = bgnbd_preds.filter(
            pl.col('customer_id') == row['customer_id']
        )['probability_alive'][0] if 'probability_alive' in bgnbd_preds.columns else None
        
        score_records.append({
            'customer_id':         row['customer_id'],
            'model_version':       FUSION_MODEL_VERSION,
            'scored_at':           scored_at,
            'ltv_source':          'full_model',
            'ltv_12m':             row['ltv_12m'],
            'ltv_24m':             row['ltv_24m'],
            'ltv_36m':             row['ltv_36m'],
            'bgnbd_ltv_12m':       row.get('bgnbd_ltv_12m'),
            'bgnbd_ltv_36m':       row.get('bgnbd_ltv_36m'),
            'transformer_ltv_12m': row.get('transformer_ltv_12m'),
            'transformer_ltv_36m': row.get('transformer_ltv_36m'),
            'meta_weight_bgnbd':   row.get('meta_weight_bgnbd'),
            'meta_weight_transformer': row.get('meta_weight_transformer'),
            'probability_alive_12m': float(p_alive) if p_alive is not None else None,
            'segment':             row['segment'],
            'ltv_percentile':      row['ltv_percentile'],
            'recommended_max_cac': row['recommended_max_cac'],
        })

    n_scores = db.bulk_upsert(
        'final_ltv_scores', score_records,
        conflict_columns=['customer_id', 'model_version'],
        batch_size=500
    )
    print(f'✓ {n_scores:,} final LTV scores saved')
    
    # 3. Save SHAP global importance
    if not global_importance.is_empty():
        print('Saving SHAP importance...')
        shap_records = global_importance.with_columns(
            pl.lit(FUSION_MODEL_VERSION).alias('model_version'),
            pl.lit(datetime.now(timezone.utc).isoformat()).alias('computed_at'),
        ).to_dicts()
        db.bulk_upsert('shap_global_importance', shap_records,
                       conflict_columns=['model_version', 'feature_name'])
        print(f'✓ SHAP importance saved')
    
    # 4. Save segment boundaries
    print('Saving segment boundaries...')
    seg_bounds = compute_segment_boundaries(final_segmented, FUSION_MODEL_VERSION)
    db.bulk_upsert('ltv_segment_boundaries', [seg_bounds],
                   conflict_columns=['model_version'])
    
    # 5. Save model to disk
    print('Saving fusion model to disk...')
    fusion.save_to_disk(MODELS_DIR)
    
    # 6. Log pipeline run
    db.bulk_upsert('pipeline_runs', [{
        'run_id':            run_id,
        'pipeline_name':     'fusion_scoring',
        'status':            'success',
        'started_at':        datetime.now(timezone.utc).isoformat(),
        'records_processed': n_scores,
        'metadata': {
            'model_version': FUSION_MODEL_VERSION,
            'gini':          metrics_for_logging.get('gini_coefficient'),
            'top_lift':      metrics_for_logging.get('top_decile_lift'),
        },
        'wandb_run_id': wandb_run_id,
    }], conflict_columns=['run_id'])

    print(f'\n✓ Week 5 complete. run_id={run_id}')

2026-05-01 23:29:46.063 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/1 into fusion_model_registry (1 rows)


Saving fusion model registry...


2026-05-01 23:29:46.121 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 1 rows into fusion_model_registry
2026-05-01 23:29:46.123 | INFO     | backend.ml.fusion:save_registry:678 - Fusion model registry saved — fusion_v1


Saving final LTV scores...


2026-05-01 23:30:11.817 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/6 into final_ltv_scores (500 rows)
2026-05-01 23:30:36.824 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 2/6 into final_ltv_scores (500 rows)
2026-05-01 23:31:01.490 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 3/6 into final_ltv_scores (500 rows)
2026-05-01 23:31:26.678 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 4/6 into final_ltv_scores (500 rows)
2026-05-01 23:31:51.218 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 5/6 into final_ltv_scores (500 rows)
2026-05-01 23:32:01.480 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 6/6 into final_ltv_scores (208 rows)
2026-05-01 23:32:01.518 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 2708 rows into final_ltv_scores


✓ 2,708 final LTV scores saved
Saving SHAP importance...


2026-05-01 23:32:02.658 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/1 into shap_global_importance (13 rows)
2026-05-01 23:32:02.714 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 13 rows into shap_global_importance
2026-05-01 23:32:02.862 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/1 into ltv_segment_boundaries (1 rows)
2026-05-01 23:32:02.898 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 1 rows into ltv_segment_boundaries


✓ SHAP importance saved
Saving segment boundaries...
Saving fusion model to disk...


2026-05-01 23:32:02.930 | INFO     | backend.ml.fusion:save_to_disk:597 - Fusion model saved → E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\models
2026-05-01 23:32:03.068 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/1 into pipeline_runs (1 rows)
2026-05-01 23:32:03.105 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 1 rows into pipeline_runs



✓ Week 5 complete. run_id=59c95d7e


In [ ]:
print('\n=== Week 5 Summary ===')
print(f'  Fusion model version:  {FUSION_MODEL_VERSION}')
print(f'  Customers scored:      {len(final_segmented):,}')
print(f'  MAE LTV 12m:           £{val_metrics["mae_ltv_12m"]:.2f}  ({100*val_metrics["mae_pct_12m"]:.1f}%)')
print(f'  Gini:                  {val_metrics["gini_coefficient"]:.4f}')
print(f'  Top decile lift:       {val_metrics["top_decile_lift"]:.2f}×')
print(f'  Calibration error:     {val_metrics["calibration_error"]:.4f}')
print(f'  Mean LTV 36m:          £{final_segmented["ltv_36m"].mean():.2f}')
print(f'\nSegments:')
for seg in ['champions', 'high_value', 'medium_value', 'low_value']:
    n = (final_segmented['segment'] == seg).sum()
    pct = 100 * n / len(final_segmented)
    print(f'  {seg:15s}: {n:>5} ({pct:.1f}%)')



=== Week 5 Summary ===
  Fusion model version:  fusion_v1
  Customers scored:      2,708
  MAE LTV 12m:           £846.68  (65.9%)
  Gini:                  0.8132
  Top decile lift:       5.55×
  Calibration error:     0.6156
  Mean LTV 36m:          £1031.15

Segments:
  champions      :    30 (1.1%)
  high_value     :    62 (2.3%)
  medium_value   :   418 (15.4%)
  low_value      :  2198 (81.2%)

Next: backend/api/main.py (Phase 6 — Week 6)
